# Silver data cleaning: open charge FI

In [ ]:
import pandas as pd
from pathlib import Path
import openpyxl
import geopandas as gpd
import shapely
import pyogrio

print(f"openpyxl version: {openpyxl.__version__}") # To read the municipality Excel file
print(f"gpd version: {gpd.__version__}") # The main library for reading GeoPackages and performing spatial joins
print(f"shapely version: {shapely.__version__}") # Used by GeoPandas for creating and working with geometries (e.g. points_from_xy())
print(f"pyogrio version: {pyogrio.__version__}") # Fast and recommended I/O backend for GeoPandas. It handles .gpkg files efficiently.

print(f"Current directory: {Path.cwd()}")

## Bronze -> Silver transformation

In [ ]:
# Read the Bronze-level data, the CSV file into a DataFrame
bronze_df = pd.read_csv("../Bronze/open_charge_raw_FI.csv", encoding="utf-8")

# Inspect the DataFrame
print(bronze_df.shape, "\n")
# print(bronze_df.columns, "\n")
print(bronze_df.dtypes)

In [ ]:
# Define the list of columns to keep and mapping, because we want to rename some columns
column_mapping = {
    "ID": "id",
    "NumberOfPoints": "number_of_points",
    "DateCreated": "date_created", # this one needs to be split later, only one key can exist in a Python dict
    "StatusType.IsOperational": "is_operational",
    "AddressInfo.Town": "town_api", # original API value, contains 1547 invalid municipalities → not usable
    "AddressInfo.StateOrProvince": "region_api", # original API value, contains 826 invalid regions → not usable
    "AddressInfo.Country.Title": "country",
    "AddressInfo.Latitude": "latitude",
    "AddressInfo.Longitude": "longitude",
    "fetch_timestamp": "fetch_timestamp"
}

# Select and rename
silver_df = (
    bronze_df[list(column_mapping.keys())]
    .rename(columns=column_mapping)
)

# Convert types
silver_df["date_created"] = pd.to_datetime(silver_df["date_created"])

# Derive columns
silver_df = silver_df.assign(
    year=silver_df["date_created"].dt.year,
    month=silver_df["date_created"].dt.month,
)

# Read the municipalities and regions from the official Excel
# See https://stat.fi/en/luokitukset/tupa -> "Municipalities and Regional Divisions Based on Municipalities 2026"
municipalities_and_regions_df = pd.read_excel(
    "en26_Municipalities_and_Regional_Divisions_Based_on_Municipalities_2026_in_Finnish,_Swedish_and_English.xlsx",
    usecols="A,B,H",
    skiprows=2,
    header=None,
    names=[
        "municipality",
        "municipality_code", # needed later for joining
        "region"
    ]
)

# Handle a special case in the Excel: "Maarianhamina - Mariehamn" → "Maarianhamina"
municipalities_and_regions_df["municipality"] = (
    municipalities_and_regions_df["municipality"]
    .replace({
        "Maarianhamina - Mariehamn": "Maarianhamina"
    })
)

display(municipalities_and_regions_df.head())
print(municipalities_and_regions_df.shape)
print(municipalities_and_regions_df.dtypes)

# Define the final column order explicitly
silver_df = silver_df[
    [
        "id",
        "number_of_points",
        "year",
        "month",
        "date_created",
        "is_operational",
        "town_api", # included for debugging purposes, will be dropped later
        "region_api", # included for debugging purposes, will be dropped later
        "country",
        "latitude",
        "longitude",
        "fetch_timestamp",
    ]
]

# Inspect the silver DataFrame
print(silver_df.shape, "\n")
print(silver_df.columns, "\n")
print(silver_df.dtypes)

In [ ]:
# Convert the data types to use the nullable Pandas dtypes (Int64, boolean, string) for Silver-layer data
# because they preserve missing values (<NA>)

silver_df["number_of_points"] = silver_df["number_of_points"].astype("Int64")

silver_df["year"] = silver_df["year"].astype("Int64")
silver_df["month"] = silver_df["month"].astype("Int64")

# print(silver_df["is_operational"].unique()) # [True, <NA>, False]
silver_df["is_operational"] = silver_df["is_operational"].astype("boolean")

# use string of Pandas instead of str
string_columns = ["town_api", "region_api", "country", "fetch_timestamp"]
silver_df[string_columns] = silver_df[string_columns].astype("string")

# use datetime for the timestamp of fetching
silver_df["fetch_timestamp"] = pd.to_datetime(silver_df["fetch_timestamp"], errors="coerce", utc=True)

# Summary of data types:
# Int64 for nullable integers
# boolean for nullable booleans
# string for text
# float64 for coordinates
# datetime for date_created and fetch_timestamp

print(silver_df.dtypes)

### Use GeoPandas to find the correct municipality

#### Do a spatial join with the official municipality polygons (town_api -> municipality)

In [ ]:
# Source: National Land Survey of Finland
# https://www.maanmittauslaitos.fi/en/maps-and-spatial-data/datasets-and-interfaces/product-descriptions/division-administrative-areas-vector ->
# https://asiointi.maanmittauslaitos.fi/karttapaikka/tiedostopalvelu?lang=en ->
# https://asiointi.maanmittauslaitos.fi/karttapaikka/tiedostopalvelu/hallinnolliset_aluejaot_vektori?lang=en ->
# Select product: Division into administrative areas 1:10 000

# Read the GeoPackage
municipalities_gdf = gpd.read_file(
    "SuomenHallinnollisetKuntajakopohjaisetAluejaot_2026_10k.gpkg"
)

print(municipalities_gdf.shape)
display(municipalities_gdf.head())

# Inspect the available layers
layers = gpd.list_layers(
    "SuomenHallinnollisetKuntajakopohjaisetAluejaot_2026_10k.gpkg"
)

display(layers)

# Read only the municipality layer
municipalities_gdf = gpd.read_file(
    "SuomenHallinnollisetKuntajakopohjaisetAluejaot_2026_10k.gpkg",
    layer="kunta"
)

# display(municipalities_gdf.head())
municipalities_gdf.info()

# Inspect the columns
print(municipalities_gdf.columns.tolist())

# Convert charging stations to points
municipalities_gdf = municipalities_gdf[
    ["natcode", "namefin", "nameswe", "geometry"]
].rename(columns={
    "natcode": "municipality_code",
    "namefin": "municipality_fi",
    "nameswe": "municipality_sv"
})

stations_gdf = gpd.GeoDataFrame(
    silver_df,
    geometry=gpd.points_from_xy(
        silver_df["longitude"],
        silver_df["latitude"]
    ),
    crs="EPSG:4326"
)

# make sure both GeoDataFrames use the same CRS
if municipalities_gdf.crs is None:
    raise ValueError("municipalities_gdf has no CRS defined.")

stations_gdf = stations_gdf.to_crs(municipalities_gdf.crs)

# Spatial join
stations_with_municipality = gpd.sjoin(
    stations_gdf,
    municipalities_gdf,
    how="left",
    predicate="within"
)

# Inspect the result
stations_with_municipality[
    [
        "id",
        "town_api",
        "region_api",
        "municipality_code",
        "municipality_fi",
        "municipality_sv",
        "latitude",
        "longitude"
    ]
].tail(20)

### Use a lookup table to add the official region

In [ ]:
# 1. Prepare municipality polygons
# Make municipality polygon columns ready for spatial join
municipalities_lookup_gdf = (
    municipalities_gdf[
        ["municipality_code", "municipality_fi", "municipality_sv", "geometry"]
    ]
    .rename(columns={
        "municipality_code": "lookup_municipality_code",
        "municipality_fi": "lookup_municipality_fi",
        "municipality_sv": "lookup_municipality_sv",
    })
)

# 2. Create point geometries from silver_df
stations_gdf = gpd.GeoDataFrame(
    silver_df,
    geometry=gpd.points_from_xy(
        silver_df["longitude"],
        silver_df["latitude"]
    ),
    crs="EPSG:4326"
)

if municipalities_lookup_gdf.crs is None:
    raise ValueError("municipalities_lookup_gdf has no CRS defined.")

stations_gdf = stations_gdf.to_crs(municipalities_lookup_gdf.crs)

# 3. Spatial join: coordinates -> official municipality
stations_with_municipality = gpd.sjoin(
    stations_gdf,
    municipalities_lookup_gdf,
    how="left",
    predicate="within"
)

# Rename to the final Silver schema
silver_df = (
    pd.DataFrame(
        stations_with_municipality.drop(
            columns=["geometry", "index_right"],
            errors="ignore"
        )
    )
    .rename(columns={
        "lookup_municipality_code": "municipality_code",
        "lookup_municipality_fi": "municipality",
        "lookup_municipality_sv": "municipality_sv",
    })
)

# Add the region from the official mapping using municipality_code

# Make sure both municipality_code columns have the same dtype and format
silver_df["municipality_code"] = (
    silver_df["municipality_code"]
    .astype("string")
    .str.zfill(3)
)

municipalities_and_regions_df["municipality_code"] = (
    municipalities_and_regions_df["municipality_code"]
    .astype("string")
    .str.zfill(3)
)

# Keep only the lookup columns needed for the merge
region_lookup_df = (
    municipalities_and_regions_df[
        ["municipality_code", "region"]
    ]
    .drop_duplicates()
)

# Merge by municipality_code, not municipality name
silver_df = silver_df.drop(columns=["region"], errors="ignore")

silver_df = silver_df.merge(
    region_lookup_df,
    on="municipality_code",
    how="left"
)

# After this, silver_df["region"] comes from the official municipality → region mapping, not from Bronze.

## Silver quality checks

In [ ]:
# Business rules and quality checks

# Match the data with the Traficom dataset that is until the end of 2026-05
REFERENCE_YEAR = 2026
REFERENCE_MONTH = 5

# An approximate bounding box for mainland Finland (including Åland and Lapland)
# Latitude: 59.5° to 70.5°
# Longitude: 19.0° to 32.0°
FINLAND_LAT_MIN = 59.5
FINLAND_LAT_MAX = 70.5
FINLAND_LON_MIN = 19.0
FINLAND_LON_MAX = 32.0

silver_df["rejection_reason"] = pd.Series(
    pd.NA,
    index=silver_df.index,
    dtype="string"
)

# Convert recently added columns to Pandas' nullable extension dtypes
string_columns = ["municipality", "municipality_sv", "region"]
silver_df[string_columns] = silver_df[string_columns].astype("string")

quality_issues = []
total_rows = len(silver_df)

# Helper function for adding failed checks
def add_quality_issue(check, details, failed_rows=None, **extra_fields):
    quality_issues.append({
        "check": check,
        "status": "FAILED",
        "details": details,
        "total_rows": total_rows,
        **extra_fields
    })

    # Append rejection reason to affected rows (store all reasons, separated by semicolons)
    if failed_rows is not None:
        current = silver_df.loc[failed_rows, "rejection_reason"].fillna("")

        silver_df.loc[failed_rows, "rejection_reason"] = (
            current.where(
                current.eq(""),
                current + "; "
            ) + check
        )


################
# Business rules
################

# B1. Åland not allowed, because the Traficom data used later in this project does not include registrations in Åland
excluded_region_rows = silver_df[
    silver_df["region"] == "Åland"
]

if not excluded_region_rows.empty:
    add_quality_issue(
        check="REGION_NOT_INCLUDED",
        details=f"Rows from excluded region Åland: {len(excluded_region_rows)}",
        affected_rows=len(excluded_region_rows),
        invalid_pct=round(len(excluded_region_rows) / total_rows * 100, 2),
        failed_rows=excluded_region_rows.index
    )


# B2. Exclude charging stations added after 2026-05
future_rows = silver_df[
    (silver_df["year"] > REFERENCE_YEAR)
    | (
        (silver_df["year"] == REFERENCE_YEAR)
        & (silver_df["month"] > REFERENCE_MONTH)
    )
]

if not future_rows.empty:
    add_quality_issue(
        check="DATE_AFTER_REFERENCE_PERIOD",
        details=f"Charging stations added after 2026-05: {len(future_rows)}",
        affected_rows=len(future_rows),
        invalid_pct=round(len(future_rows) / total_rows * 100, 2),
        failed_rows=future_rows.index
    )



################
# Quality checks
################


# 1. Row count check
if len(silver_df) != len(bronze_df):
    add_quality_issue(
        check="ROW_COUNT",
        details=f"Bronze rows: {len(bronze_df)}, Silver rows: {len(silver_df)}",
        bronze_rows=len(bronze_df),
        silver_rows=len(silver_df)
    )


# 2. Duplicate check by id
duplicate_ids = silver_df[silver_df.duplicated(subset=["id"], keep=False)]

if not duplicate_ids.empty:
    add_quality_issue(
        check="DUPLICATE_IDS",
        details=f"Duplicate IDs found: {duplicate_ids['id'].nunique()}",
        affected_rows=len(duplicate_ids),
        duplicate_id_count=duplicate_ids["id"].nunique(),
        failed_rows=duplicate_ids.index
    )


# 3. Missing municipality enrichment
missing_municipality_rows = silver_df[silver_df["municipality"].isna()]

if not missing_municipality_rows.empty:
    add_quality_issue(
        check="MISSING_MUNICIPALITY_ENRICHMENT",
        details=f"Missing municipality after enrichment: {len(missing_municipality_rows)}",
        affected_rows=len(missing_municipality_rows),
        invalid_pct=round(len(missing_municipality_rows) / total_rows * 100, 2),
        failed_rows=missing_municipality_rows.index
    )


# 4. Missing region enrichment
# Since region is derived from official municipality data, there are really only two possibilities:
# 1. The enrichment succeeded → region is guaranteed to be valid.
# 2. The enrichment failed → region is NaN.

# simply check for missing values
missing_region_rows = silver_df[silver_df["region"].isna()]

if not missing_region_rows.empty:
    add_quality_issue(
        check="MISSING_REGION_ENRICHMENT",
        details=f"Missing region after enrichment: {len(missing_region_rows)}",
        affected_rows=len(missing_region_rows),
        invalid_pct=round(len(missing_region_rows) / total_rows * 100, 2),
        failed_rows=missing_region_rows.index
    )


# 5. Invalid numeric values: coordinates
invalid_coordinates = silver_df[
    silver_df["latitude"].lt(FINLAND_LAT_MIN)
    | silver_df["latitude"].gt(FINLAND_LAT_MAX)
    | silver_df["longitude"].lt(FINLAND_LON_MIN)
    | silver_df["longitude"].gt(FINLAND_LON_MAX)
]

if not invalid_coordinates.empty:
    add_quality_issue(
        check="INVALID_COORDINATES",
        details=f"Coordinates outside Finland: {len(invalid_coordinates)}",
        affected_rows=len(invalid_coordinates),
        invalid_pct=round(len(invalid_coordinates) / total_rows * 100, 2),
        failed_rows=invalid_coordinates.index
    )


# 6. Invalid numeric values: number_of_points
invalid_number_of_points = silver_df[silver_df["number_of_points"].lt(0)]

if not invalid_number_of_points.empty:
    add_quality_issue(
        check="INVALID_NUMBER_OF_POINTS",
        details=f"Invalid number_of_points rows: {len(invalid_number_of_points)}",
        affected_rows=len(invalid_number_of_points),
        invalid_pct=round(len(invalid_number_of_points) / total_rows * 100, 2),
        failed_rows=invalid_number_of_points.index
    )


# 7. Invalid municipality
valid_municipalities = set(
    municipalities_and_regions_df["municipality"]
    .dropna()
    .astype("string")
    .str.strip()
    .str.casefold()
)

valid_regions = set(
    municipalities_and_regions_df["region"]
    .dropna()
    .astype("string")
    .str.strip()
    .str.casefold()
)

invalid_municipalities = silver_df[
    silver_df["municipality"].notna()
    & ~silver_df["municipality"]
        .str.strip()
        .str.casefold()
        .isin(valid_municipalities)
].copy()

if not invalid_municipalities.empty:
    add_quality_issue(
        check="INVALID_MUNICIPALITY",
        details=f"Invalid municipality rows: {len(invalid_municipalities)}",
        affected_rows=len(invalid_municipalities),
        invalid_pct=round(len(invalid_municipalities) / total_rows * 100, 2),
        failed_rows=invalid_municipalities.index
    )


# Final quality report
quality_report = pd.DataFrame(quality_issues).fillna("") # replace NaN with blank cells so that it is more human-readable

if quality_report.empty:
    print("All quality checks passed.")
else:
    display(quality_report)

### Split the data

In [ ]:
# Define the desired column order
silver_columns = [
    "id",
    "number_of_points",
    "year",
    "month",
    "is_operational",
    "municipality",
    "region",
    "country",
    "latitude",
    "longitude",
    "rejection_reason",
    "fetch_timestamp",
]

# Split valid and rejected rows
silver_valid = (
    silver_df[silver_df["rejection_reason"].isna()]
    .loc[:, silver_columns]
    .copy()
)

silver_rejected = (
    silver_df[silver_df["rejection_reason"].notna()]
    .loc[:, silver_columns]
    .copy()
)

print("silver_valid:")
print(f"Row count: {len(silver_valid)}")
display(silver_valid.head(10))

print("silver_rejected:")
print(f"Row count: {len(silver_rejected)}")
display(silver_rejected)

## Save Silver-level data to CSV files

In [ ]:
silver_valid.to_csv("silver_ev_charging_stations_FI.csv", index=False)
silver_rejected.to_csv("REJECTED_silver_ev_charging_stations_FI.csv", index=False)